In [0]:
%run ./00_config

In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")
dbutils.widgets.text("zone", "")
dbutils.widgets.text("run_id", "")
START_DATE = dbutils.widgets.get("start_date")
END_DATE = dbutils.widgets.get("end_date")
ZONE = dbutils.widgets.get("zone")
RUN_ID = dbutils.widgets.get("run_id")

In [0]:
def apply_time_zone_filter(df, ts_col="event_time_utc", zone_col="zone"):
    out = df
    if START_DATE and END_DATE:
        out = out.filter(
            (F.col(ts_col) >= F.to_timestamp(F.lit(START_DATE))) &
            (F.col(ts_col) < F.to_timestamp(F.lit(END_DATE)))
        )
    if ZONE:
        out = out.filter(F.upper(F.col(zone_col)) == ZONE.upper())
    return out

In [0]:
from pyspark.sql import functions as F
# Generation dependency
gen = spark.table(T_GEN_WIDE)
gold_dep = (
    gen
    .withColumn("renewable_generation_mw",
        F.coalesce(F.col("biomass"), F.lit(0.0)) +
        F.coalesce(F.col("hydro_run_of_river_and_pondage"), F.lit(0.0)) +
        F.coalesce(F.col("wind_onshore"), F.lit(0.0)) +
        F.coalesce(F.col("wind_offshore"), F.lit(0.0)) +
        F.coalesce(F.col("solar"), F.lit(0.0))
    )
    .withColumn("non_renewable_generation_mw",
        F.coalesce(F.col("fossil_hard_coal"), F.lit(0.0)) +
        F.coalesce(F.col("fossil_gas"), F.lit(0.0)) +
        F.coalesce(F.col("nuclear"), F.lit(0.0)) +
        F.coalesce(F.col("waste"), F.lit(0.0)) +
        F.coalesce(F.col("other"), F.lit(0.0))
    )
    .withColumn("total_generation_mw", F.coalesce(F.col("total_generation_mw_avg"), F.lit(0.0)))
    .withColumn("renewable_share_pct", F.when(F.col("total_generation_mw") > 0, F.col("renewable_generation_mw") / F.col("total_generation_mw")).otherwise(F.lit(0.0)))
    .withColumn("non_renewable_share_pct", F.when(F.col("total_generation_mw") > 0, F.col("non_renewable_generation_mw") / F.col("total_generation_mw")).otherwise(F.lit(0.0)))
    .select("event_time_utc", "zone", "renewable_generation_mw", "non_renewable_generation_mw", "total_generation_mw", "renewable_share_pct", "non_renewable_share_pct")
)

gold_dep = apply_time_zone_filter(gold_dep)
gold_dep.write.mode("overwrite").format("delta").saveAsTable(T_GOLD_DEP)

In [0]:
#h2.gold.renewable_mix_hourly
from pyspark.sql import functions as F
gen = spark.table(T_GEN_WIDE)
renewable_mix = (
    gen
    .withColumn("biomass_mw", F.coalesce(F.col("biomass"), F.lit(0.0)))
    .withColumn("hydro_mw", F.coalesce(F.col("hydro_run_of_river_and_pondage"), F.lit(0.0)))
    .withColumn("wind_onshore_mw", F.coalesce(F.col("wind_onshore"), F.lit(0.0)))
    .withColumn("wind_offshore_mw", F.coalesce(F.col("wind_offshore"), F.lit(0.0)))
    .withColumn("solar_mw", F.coalesce(F.col("solar"), F.lit(0.0)))
    .withColumn(
        "renewable_generation_mw",
        F.col("biomass_mw") + F.col("hydro_mw") + F.col("wind_onshore_mw") + F.col("wind_offshore_mw") + F.col("solar_mw")
    )
    .withColumn("total_generation_mw", F.coalesce(F.col("total_generation_mw_avg"), F.lit(0.0)))
    .withColumn(
        "renewable_share_pct",
        F.when(F.col("total_generation_mw") > 0, F.col("renewable_generation_mw") / F.col("total_generation_mw")).otherwise(F.lit(0.0))
    )
    .select(
        "event_time_utc",
        "zone",
        "biomass_mw",
        "hydro_mw",
        "wind_onshore_mw",
        "wind_offshore_mw",
        "solar_mw",
        "renewable_generation_mw",
        "total_generation_mw",
        "renewable_share_pct"
    )
)
renewable_mix = apply_time_zone_filter(renewable_mix)
renewable_mix.write.mode("overwrite").format("delta").saveAsTable(T_GOLD_RENEWABLE_MIX)

In [0]:
#h2.gold.fossil_dependency_hourly
from pyspark.sql import functions as F
gen = spark.table(T_GEN_WIDE)
fossil_dependency = (
    gen
    .withColumn("fossil_gas_mw", F.coalesce(F.col("fossil_gas"), F.lit(0.0)))
    .withColumn("fossil_hard_coal_mw", F.coalesce(F.col("fossil_hard_coal"), F.lit(0.0)))
    .withColumn("nuclear_mw", F.coalesce(F.col("nuclear"), F.lit(0.0)))
    .withColumn("waste_mw", F.coalesce(F.col("waste"), F.lit(0.0)))
    .withColumn("other_mw", F.coalesce(F.col("other"), F.lit(0.0)))
    .withColumn("fossil_generation_mw", F.col("fossil_gas_mw") + F.col("fossil_hard_coal_mw"))
    .withColumn(
        "non_renewable_generation_mw",
        F.col("fossil_generation_mw") + F.col("nuclear_mw") + F.col("waste_mw") + F.col("other_mw")
    )
    .withColumn("total_generation_mw", F.coalesce(F.col("total_generation_mw_avg"), F.lit(0.0)))
    .withColumn(
        "fossil_share_pct",
        F.when(F.col("total_generation_mw") > 0, F.col("fossil_generation_mw") / F.col("total_generation_mw")).otherwise(F.lit(0.0))
    )
    .withColumn(
        "non_renewable_share_pct",
        F.when(F.col("total_generation_mw") > 0, F.col("non_renewable_generation_mw") / F.col("total_generation_mw")).otherwise(F.lit(0.0))
    )
    .select(
        "event_time_utc",
        "zone",
        "fossil_gas_mw",
        "fossil_hard_coal_mw",
        "fossil_generation_mw",
        "non_renewable_generation_mw",
        "total_generation_mw",
        "fossil_share_pct",
        "non_renewable_share_pct"
    )
)
fossil_dependency = apply_time_zone_filter(fossil_dependency)
fossil_dependency.write.mode("overwrite").format("delta").saveAsTable(T_GOLD_FOSSIL_DEP)

In [0]:
#h2.gold.model_scoring_base
from pyspark.sql import functions as F
mw = spark.table(T_GOLD_MARKET_WEATHER)
dep = spark.table(T_GOLD_DEP)
model_scoring_base = (
    mw.alias("m")
    .join(dep.alias("d"), ["event_time_utc", "zone"], "left")
    .select(
        "event_time_utc",
        "zone",
        "price_eur_mwh",
        "avg_actual_total_load_mw",
        "day_ahead_total_load_forecast_mw",
        "temperature_c",
        "u10_ms",
        "v10_ms",
        "wind_speed_ms",
        "surface_pressure_pa",
        "renewable_generation_mw",
        "non_renewable_generation_mw",
        "renewable_share_pct",
        "non_renewable_share_pct",
        "hour_of_day",
        "day_of_week",
        "month"
    )
)
model_scoring_base = apply_time_zone_filter(model_scoring_base)
model_scoring_base.write.mode("overwrite").format("delta").saveAsTable(T_GOLD_MODEL_SCORING)

In [0]:
# Station summary
stations = spark.table(T_STATIONS_CLEAN)
if ZONE:
    stations = stations.filter(F.upper(F.col("country")) == ZONE.upper())
station_summary = (
    stations.groupBy(F.col("country").alias("zone"))
    .agg(
        F.count("*").alias("stations_total"),
        F.sum(F.when(F.col("status_id") == 1, 1).otherwise(0)).alias("stations_in_operation"),
        F.sum(F.when(F.col("status_id") == 2, 1).otherwise(0)).alias("stations_planned"),
        F.sum(F.when(F.col("status_id") == 3, 1).otherwise(0)).alias("stations_old_projects")
    )
    .withColumn("active_ratio", F.when(F.col("stations_total") > 0, F.col("stations_in_operation") / F.col("stations_total")).otherwise(F.lit(0.0)))
)
station_summary.write.mode("overwrite").format("delta").saveAsTable(T_GOLD_STATIONS)

In [0]:


# Market weather hourly
spark.table(T_TRAINING_BASE).write.mode("overwrite").format("delta").saveAsTable(T_GOLD_MARKET_WEATHER)
# QA
spark.table(T_GOLD_DEP).selectExpr("count(*) rows", "avg(renewable_share_pct) avg_ren_share").show()
spark.table(T_GOLD_STATIONS).show()